# 00 — Data Exploration

Explore the NIH ChestX-ray14 dataset before training.

**Covers**: label distribution, class imbalance, image statistics, sample visualisation.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from src.data.extract import prepare_data, load_labels_dataframe
from src.utils.config import PATHOLOGY_CLASSES, Paths
print('Setup OK')

## 2. Load Labels

In [ ]:
# Download labels CSV if not present
from src.data.extract import download_labels_csv
download_labels_csv()

df = load_labels_dataframe()
print(f'Total images: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 3. Class Distribution (Imbalance)

In [ ]:
counts = df[PATHOLOGY_CLASSES].sum().sort_values(ascending=False)
pct    = (counts / len(df) * 100).round(2)

fig = go.Figure(go.Bar(
    x=counts.values, y=counts.index, orientation='h',
    marker_color='steelblue',
    text=[f'{c:,} ({p}%)' for c,p in zip(counts.values, pct.values)],
    textposition='outside'
))
fig.update_layout(
    title='Class distribution in NIH ChestX-ray14',
    yaxis={'autorange': 'reversed'}, height=450, margin=dict(l=10, r=120)
)
fig.show()
print('Most common: No Finding (~50% of dataset)')

## 4. Multi-label Co-occurrence

In [ ]:
# How often do pathologies appear together?
from itertools import combinations
import numpy as np

cooc = np.zeros((len(PATHOLOGY_CLASSES), len(PATHOLOGY_CLASSES)))
for i, ci in enumerate(PATHOLOGY_CLASSES):
    for j, cj in enumerate(PATHOLOGY_CLASSES):
        cooc[i, j] = (df[ci] & df[cj]).sum()

fig = px.imshow(
    cooc, x=PATHOLOGY_CLASSES, y=PATHOLOGY_CLASSES,
    title='Pathology co-occurrence matrix',
    color_continuous_scale='Blues', text_auto=True,
)
fig.update_layout(height=500)
fig.show()

## 5. Sample Images

In [ ]:
from src.utils.image_utils import load_xray, get_image_stats
from src.utils.config import PATHOLOGY_CLASSES
import matplotlib.pyplot as plt

# Show sample images for each class (requires images downloaded)
image_dir = Paths.images_dir()
for cls in PATHOLOGY_CLASSES[:4]:
    sample_ids = df[df[cls]==1]['image_id'].head(3).tolist()
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    fig.suptitle(cls, fontsize=13)
    for ax, img_id in zip(axes, sample_ids):
        img_path = image_dir / img_id
        if img_path.exists():
            ax.imshow(load_xray(img_path), cmap='gray')
            ax.set_title(img_id[:15], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()